In [1]:
import pandas as pd
from tqdm.auto import tqdm
import sys
sys.path.append("../")
tqdm.pandas()

In [2]:
from src.repo_utils import clone_and_extract_tree

In [3]:
# gpt 5.2's context window is 400k tokens, leave margin for prompt + tree
CONTEXT_WINDOW = 380000

In [4]:
df_repos = pd.read_parquet("../data/02_paper_assessment/references_paper_assessment_pred.parquet.br")

In [5]:
df_repos["repo_url"].value_counts()

repo_url
Appendix                                                                                            75
https://github.com/opensafely/amr-uom-brit                                                           2
https://github.com/rutgervandeleur/ecgxai                                                            2
https://github.com/pennsignals/eol-onc                                                               2
https://github.com/htx-r/Reproduce-results-from-papers                                               2
                                                                                                    ..
https://osf.io/6y8fs/                                                                                1
https://github.com/mertkarabacak/NCDB-G2G3_Glioma                                                    1
https://github.com/ohdsi-studies/lungCancerPrognostic                                                1
https://github.com/WenjuanW/Risk_Prediction_of_Post-Stroke_30-da

In [6]:
# subset rows where repo_url is present and not the literal "Appendix" (case-insensitive)
df_repos_assessment = df_repos[
    df_repos["repo_url"].notna() &
    (df_repos["is_match"] == True) &
    (df_repos["repo_url"].astype(str).str.strip().str.lower() != "appendix")
].copy()

In [7]:
def wrapper_clone_and_extract_tree(repo_url):
    result = clone_and_extract_tree(
        repo_url=repo_url,
        context_window=CONTEXT_WINDOW,
        clone_dir="../data/03_repo_assessment/cloned_repos",
    )
    return {
        "repo_content": result.output,
        "repo_status": result.status,
        "repo_error": result.error,
    }

In [8]:
df_repos_assessment[["repo_content", "repo_status", "repo_error"]] = (
    df_repos_assessment["repo_url"]
    .progress_apply(wrapper_clone_and_extract_tree)
    .apply(pd.Series)
)

  0%|          | 0/447 [00:00<?, ?it/s]

In [12]:
df_repos_assessment.repo_status.value_counts()

repo_status
RepoStatus.OK               380
RepoStatus.INACCESSIBLE      38
RepoStatus.NOT_SUPPORTED     27
RepoStatus.EMPTY              2
Name: count, dtype: int64

In [10]:
# removing column generation as we'll run through llm again
df_repos_assessment = df_repos_assessment.drop(columns=["generation"])

In [13]:
df_repos_assessment.to_parquet(
    "../data/03_repo_assessment/references_with_repo.parquet.br",
    compression="brotli",
)